## Setup - load all databases into one big dataframe

In [ ]:
import sqlite3
import orjson
import os
import re
import pandas as pd

list_of_all_databases = [
    ('Dataset', 'Feature Extraction',   r'C:\Users\anonymous\Downloads\Experiment\Dataset\Feature-Extraction\data\scan_log_feature-extraction.db'),
    ('Dataset', 'Image Classification', r'C:\Users\anonymous\Downloads\Experiment\Dataset\Image-Classification\data\scan_log_image-classification.db'),
    ('Dataset', 'Image Segmentation',   r'C:\Users\anonymous\Downloads\Experiment\Dataset\Image-Segmentation\data\scan_log_image-segmentation.db'),
    ('Dataset', 'Text Classification',  r'C:\Users\anonymous\Downloads\Experiment\Dataset\text-classification\data\scan_log_text-classification.db'),
    ('Dataset', 'Text Generation',      r'C:\Users\anonymous\Downloads\Experiment\Dataset\text-generation\data\scan_log_text-generation.db'),
    ('Model',   'Feature Extraction',   r'C:\Users\anonymous\Downloads\Experiment\Model\feature-extraction\data\scan_log_feature-extraction.db'),
    ('Model',   'Image Classification', r'C:\Users\anonymous\Downloads\Experiment\Model\Image-classification\data\scan_log_image-classification.db'),
    ('Model',   'Image Segmentation',   r'C:\Users\anonymous\Downloads\Experiment\Model\Image-Segmentation\data\scan_log_image-segmentation.db'),
    ('Model',   'Text Classification',  r'C:\Users\anonymous\Downloads\Experiment\Model\Text-classification\data\scan_log_text-classification.db'),
    
    ('Model',   'Text Generation',      r'D:\Experiment\Model\text-generation\1to100000\data\scan_log_text-generation_1to100000.db'),
    ('Model',   'Text Generation',      r'D:\Experiment\Model\text-generation\100001to148000\data\scan_log_text-generation_100001to148000.db'),
    ('Model',   'Text Generation',      r'D:\Experiment\Model\text-generation\148000to278000\data\scan_log_text-generation_148000to278000.db'),
    ('Model',   'Text Generation',      r'D:\Experiment\Model\text-generation\278000to370162\data\scan_log_text-generation_278000to370162.db'),
    
    # ('HFscan',   'HF scan',      r'C:\Users\anonymous\Downloads\Experiment\Hfscan\scan_log_hf_rescan.db'),

    
]

local_scanners = {'bandit', 'opengrep', 'pysentry', 'trivy', 'pip_audit', 'modelaudit'}

for one_type, one_task, one_path in list_of_all_databases:
    if os.path.exists(one_path) == False:
        print("!! MISSING:", one_path)
    else:
        print("ok found:", one_type, "-", one_task)

all_the_repo_rows = []
all_the_malicious_details = []  

for one_type, one_task, one_full_path in list_of_all_databases:
    conn = sqlite3.connect(one_full_path)
    rows = conn.execute("SELECT repo_id, outcome, details FROM scans").fetchall()
    conn.close()

    for repo_id, outcome, details_str in rows:
        try:
            details = orjson.loads(details_str)
        except:
            details = {}

        file_results = details.get('file_results', []) or []
        has_local_flag = False
        has_hf_flag = False
        if details.get('hf_security_api') is not None:
            has_hf_flag = True
        for one_file in file_results:
            for v in (one_file.get('verdicts', []) or []):
                if v.get('flagged') != True:
                    continue
                if v.get('scanner') == 'hf_security_api':
                    has_hf_flag = True
                if v.get('scanner') in local_scanners:
                    has_local_flag = True

        has_project_flag = False
        for pv in (details.get('project_verdicts', []) or []):
            if pv.get('flagged') == True:
                has_project_flag = True

        flagged_files_list = details.get('flagged_files', []) or []
        if len(file_results) == 0 and len(flagged_files_list) > 0:
            has_hf_flag = True

        one_row = {
            'repo_id': repo_id,
            'type': one_type,
            'task_category': one_task,
            'source_db': one_full_path,
            'outcome': outcome,
            'is_malicious': (outcome == 'malicious'),
            'has_local_flag': has_local_flag,
            'has_hf_flag': has_hf_flag,
            'has_project_flag': has_project_flag,
        }
        all_the_repo_rows.append(one_row)

        if outcome == 'malicious':
            all_the_malicious_details.append((repo_id, one_type, one_task, details))

df_everything = pd.DataFrame(all_the_repo_rows)
print()
print("loaded", len(list_of_all_databases), "databases,", len(df_everything), "total repos")
print("malicious repos loaded:", len(all_the_malicious_details))

ok found: Dataset - Feature Extraction
ok found: Dataset - Image Classification
ok found: Dataset - Image Segmentation
ok found: Dataset - Text Classification
ok found: Dataset - Text Generation
ok found: Model - Feature Extraction
ok found: Model - Image Classification
ok found: Model - Image Segmentation
ok found: Model - Text Classification
ok found: Model - Text Generation
ok found: Model - Text Generation
ok found: Model - Text Generation
ok found: Model - Text Generation
ok found: HFscan - HF scan

loaded 14 databases, 548012 total repos
malicious repos loaded: 37598


In [2]:
hf_flagged_rows = df_everything[df_everything['has_hf_flag'] == True]
print("hf flagged rows (per database):", len(hf_flagged_rows))

hf_flagged_rows = hf_flagged_rows.drop_duplicates(subset='repo_id')
print("unique hf flagged repos:", len(hf_flagged_rows))

rows_for_the_csv = []
for index_number, one_row in hf_flagged_rows.iterrows():
    category_for_tag = one_row['task_category'].lower().replace(' ', '-')
    if one_row['type'] == 'Model':
        type_for_cli = 'models'
    else:
        type_for_cli = 'datasets'
    rows_for_the_csv.append({
        'repo_id': one_row['repo_id'],
        'category': category_for_tag,
        'type': type_for_cli,
        'old_outcome': one_row['outcome'],
        'old_had_local_flag': one_row['has_local_flag'],
        'source_db': one_row['source_db'],
    })

df_rescan_list = pd.DataFrame(rows_for_the_csv)
print(df_rescan_list.groupby(['type', 'category']).size())

df_rescan_list.to_csv('hf_flagged_rescan_list.csv', index=False)
print("saved", len(df_rescan_list), "repos to hf_flagged_rescan_list.csv")


hf flagged rows (per database): 4105
unique hf flagged repos: 2079
type      category            
datasets  feature-extraction         2
          text-classification       28
          text-generation           48
models    feature-extraction        58
          image-classification       6
          image-segmentation        47
          text-classification      160
          text-generation         1730
dtype: int64
saved 2079 repos to hf_flagged_rescan_list.csv


## 3.1 - Corpus and detection analysis

total scanned, total malicious, the malicious percentage, and how many repos the project-level
scanner flagged across the whole corpus.

In [3]:
df_without_hfscan = df_everything[df_everything['type'] != 'HFscan']

got_total_scanned = df_without_hfscan['repo_id'].nunique()
print("total repositories scanned (each repo once):", got_total_scanned)

df_malicious = df_without_hfscan[df_without_hfscan['is_malicious'] == True]

got_total_malicious = df_malicious['repo_id'].nunique()
print("total malicious repositories (each repo once):", got_total_malicious)

got_percent = round(got_total_malicious / got_total_scanned * 100, 1)
print("malicious percentage:", got_percent, "%")

repos_with_a_project_flag = df_without_hfscan[df_without_hfscan['has_project_flag'] == True]
got_project_flags = repos_with_a_project_flag['repo_id'].nunique()
print("repos with a positive project-level scanner finding:", got_project_flags)

total repositories scanned (each repo once): 540090
total malicious repositories (each repo once): 35361
malicious percentage: 6.5 %
repos with a positive project-level scanner finding: 0


### unique malicious repos

deduplicate repo_id among the malicious rows using to get unique count


In [4]:
got_unique_malicious = df_malicious['repo_id'].nunique()
print("unique malicious repo_id count (simple dedup on repo_id):", got_unique_malicious)

dupe_counts_among_malicious = df_malicious['repo_id'].value_counts()
dupes_only = dupe_counts_among_malicious[dupe_counts_among_malicious > 1]


unique malicious repo_id count (simple dedup on repo_id): 35361


## Table 1 - per task-category breakdown (Scanned / Malicious / Local / HF / Overlap)

From only the unique count, this rebuilds row by row from the raw databases.

In [5]:
repos_without_hfscan = df_everything[df_everything['type'] != 'HFscan']
hfscan_rows = df_everything[df_everything['type'] == 'HFscan']

hfscan_flags = hfscan_rows[hfscan_rows['is_malicious'] == True].groupby('repo_id').agg(
    rescan_local=('has_local_flag', 'max'),
    rescan_hf=('has_hf_flag', 'max'),
)
# print("rescan repos with flags:", len(hfscan_flags))

table_1_rows = []
for (one_type, one_task), group in repos_without_hfscan.groupby(['type', 'task_category']):
    crawled_repos_here = group['repo_id'].nunique()

    group_malicious = group[group['is_malicious'] == True]
    one_row_per_repo = group_malicious.groupby('repo_id').agg(
        has_local_flag=('has_local_flag', 'max'),
        has_hf_flag=('has_hf_flag', 'max'),
    )
    one_row_per_repo = one_row_per_repo.join(hfscan_flags, how='left')
    one_row_per_repo['rescan_local'] = one_row_per_repo['rescan_local'].fillna(False).astype(bool)
    one_row_per_repo['rescan_hf'] = one_row_per_repo['rescan_hf'].fillna(False).astype(bool)
    local_flag = one_row_per_repo['has_local_flag'].astype(bool) | one_row_per_repo['rescan_local']
    hf_flag = one_row_per_repo['has_hf_flag'].astype(bool) | one_row_per_repo['rescan_hf']

    table_1_rows.append({
        'Type': one_type,
        'Task Category': one_task,
        'Crawled': crawled_repos_here,
        'Malicious': len(one_row_per_repo),
        'Local': int(local_flag.sum()),
        'HF': int(hf_flag.sum()),
        'Overlap': int((local_flag & hf_flag).sum()),
    })

df_table_1 = pd.DataFrame(table_1_rows)



In [6]:
all_malicious = repos_without_hfscan[repos_without_hfscan['is_malicious'] == True]
all_one_row_per_repo = all_malicious.groupby('repo_id').agg(
    has_local_flag=('has_local_flag', 'max'),
    has_hf_flag=('has_hf_flag', 'max'),
)
all_one_row_per_repo = all_one_row_per_repo.join(hfscan_flags, how='left')
all_one_row_per_repo['rescan_local'] = all_one_row_per_repo['rescan_local'].fillna(False).astype(bool)
all_one_row_per_repo['rescan_hf'] = all_one_row_per_repo['rescan_hf'].fillna(False).astype(bool)
all_local_flag = all_one_row_per_repo['has_local_flag'].astype(bool) | all_one_row_per_repo['rescan_local']
all_hf_flag = all_one_row_per_repo['has_hf_flag'].astype(bool) | all_one_row_per_repo['rescan_hf']


total_row = pd.DataFrame([{
    'Type': 'TOTAL',
    'Task Category': '',
    'Crawled': repos_without_hfscan['repo_id'].nunique(),
    'Malicious': len(all_one_row_per_repo),
    'Local': int(all_local_flag.sum()),
    'HF': int(all_hf_flag.sum()),
    'Overlap': int((all_local_flag & all_hf_flag).sum()),
}])
df_table_1 = pd.concat([df_table_1, total_row], ignore_index=True)
df_table_1['Local Only'] = df_table_1['Local'] - df_table_1['Overlap']
df_table_1['HF Only'] = df_table_1['HF'] - df_table_1['Overlap']


# print("great!!! the check loop ran")

print(df_table_1.to_string(index=False))
df_table_1

   Type        Task Category  Crawled  Malicious  Local   HF  Overlap  Local Only  HF Only
Dataset   Feature Extraction     2736         92     90    2        0          90        2
Dataset Image Classification     4263        142    142    0        0         142        0
Dataset   Image Segmentation     1627         74     74    0        0          74        0
Dataset  Text Classification    15128        783    758   29        4         754       25
Dataset      Text Generation    23611        880    833   57       10         823       47
  Model   Feature Extraction    18236       2928   2870   58        0        2870       58
  Model Image Classification    27216        661    656    6        1         655        5
  Model   Image Segmentation     2657        301    255   47        1         254       46
  Model  Text Classification   117386       1832   1676  160        4        1672      156
  Model      Text Generation   333101      27889  26167 1730        8       26159     1722

,Type,Task Category,Crawled,Malicious,Local,HF,Overlap,Local Only,HF Only
0,Dataset,Feature Extraction,2736,92,90,2,0,90,2
1,Dataset,Image Classification,4263,142,142,0,0,142,0
2,Dataset,Image Segmentation,1627,74,74,0,0,74,0
3,Dataset,Text Classification,15128,783,758,29,4,754,25
4,Dataset,Text Generation,23611,880,833,57,10,823,47
5,Model,Feature Extraction,18236,2928,2870,58,0,2870,58
6,Model,Image Classification,27216,661,656,6,1,655,5
7,Model,Image Segmentation,2657,301,255,47,1,254,46
8,Model,Text Classification,117386,1832,1676,160,4,1672,156
9,Model,Text Generation,333101,27889,26167,1730,8,26159,1722


### Table -- the files participating in Local Only and HF Only

The overlap table above is one row per flagged file for the 24 overlap repos. This does the same thing
for the other two columns of Table 1 -- every **Local Only** repo and every **HF Only** repo -- so each
row is one flagged file with whatever flagged it.

- `local_only_flagged_files.csv` -- the files the local scanners (Bandit / OpenGrep / ModelAudit / PySentry / Trivy / pip-audit) flagged, in repos HF never flagged
- `hf_only_flagged_files.csv` -- the files HF flagged, in repos no local scanner flagged (includes the >30 GB repos where only the HF api ran)

In [ ]:
# step 1 - check which group every malicious repo is in
repos_without_hfscan_here = df_everything[df_everything['type'] != 'HFscan']
malicious_repo_ids_here = set(repos_without_hfscan_here[repos_without_hfscan_here['is_malicious'] == True]['repo_id'])

# confirm from the malicious rescan
rows_to_use_here = df_everything[(df_everything['is_malicious'] == True) & (df_everything['repo_id'].isin(malicious_repo_ids_here))]
flags_per_repo = rows_to_use_here.groupby('repo_id')[['has_local_flag', 'has_hf_flag']].max()
# print(flags_per_repo.head())

local_only_repos = set(flags_per_repo[(flags_per_repo['has_local_flag'] == True) & (flags_per_repo['has_hf_flag'] == False)].index)
hf_only_repos = set(flags_per_repo[(flags_per_repo['has_local_flag'] == False) & (flags_per_repo['has_hf_flag'] == True)].index)
print("local only repos:", len(local_only_repos))
print("hf only repos   :", len(hf_only_repos))

local only repos: 33282
hf only repos   : 2055


### Table - check the scanned table (outcomes)

In [24]:
# To see a better break down!!
outcome_priority = ['malicious', 'clean', 'no_supported_files', 'gated', 'not_found',
                    'download_timeout', 'download_error', 'error']

rows_for_outcomes = df_everything[df_everything['type'] != 'HFscan'].copy()
outcome_rank = {}
for position_number in range(len(outcome_priority)):
    outcome_rank[outcome_priority[position_number]] = position_number
rows_for_outcomes['rank'] = rows_for_outcomes['outcome'].map(outcome_rank).fillna(len(outcome_priority))
rows_for_outcomes = rows_for_outcomes.sort_values('rank', kind='stable')

one_outcome_per_repo_in_category = rows_for_outcomes.drop_duplicates(subset=['type', 'task_category', 'repo_id'], keep='first')
one_outcome_per_repo_overall = rows_for_outcomes.drop_duplicates(subset='repo_id', keep='first')

outcome_table = pd.crosstab([one_outcome_per_repo_in_category['type'], one_outcome_per_repo_in_category['task_category']],
                            one_outcome_per_repo_in_category['outcome'])

columns_in_order = []
for one_outcome in outcome_priority:
    if one_outcome in outcome_table.columns:
        columns_in_order.append(one_outcome)

for one_outcome in outcome_table.columns:
    if one_outcome not in columns_in_order:
        print("heads up -- outcome i didnt expect:", one_outcome)
        columns_in_order.append(one_outcome)
outcome_table = outcome_table[columns_in_order]

all_row_counts = one_outcome_per_repo_overall['outcome'].value_counts()
for one_outcome in columns_in_order:
    if one_outcome in all_row_counts.index:
        outcome_table.loc[('ALL', 'ALL'), one_outcome] = all_row_counts[one_outcome]
    else:
        outcome_table.loc[('ALL', 'ALL'), one_outcome] = 0

never_scanned_outcomes = ['gated', 'not_found', 'download_timeout', 'download_error', 'error']
never_scanned_columns = []
for one_outcome in never_scanned_outcomes:
    if one_outcome in outcome_table.columns:
        never_scanned_columns.append(one_outcome)

outcome_table['crawled'] = outcome_table[columns_in_order].sum(axis=1)
outcome_table['never scanned'] = outcome_table[never_scanned_columns].sum(axis=1)
outcome_table['scanned'] = outcome_table['crawled'] - outcome_table['never scanned']
outcome_table['had scannable files'] = outcome_table['clean'] + outcome_table['malicious']
outcome_table = outcome_table.astype(int)

print("'Crawled' (each repo counted only once):")
print(outcome_table.to_string())
print()
print("TOTAL crawled:", got_total_scanned)
print("TOTAL malicious:", got_total_malicious)

'Crawled' (each repo counted only once):
outcome                       malicious   clean  no_supported_files  gated  not_found  download_timeout  download_error  error  crawled  never scanned  scanned  had scannable files
type    task_category                                                                                                                                                               
Dataset Feature Extraction           92    2072                 419    132          1                 0               0     20     2736            153     2583                 2164
        Image Classification        142    2524                1315    254          0                 0               0     28     4263            282     3981                 2666
        Image Segmentation           74    1048                 433     67          1                 0               0      4     1627             72     1555                 1122
        Text Classification         783   12329       

## Attribution of malicious outcomes across detection layers

how many malicious repos were flagged by a local scanner, by HF ?

## 3.3 - Overlap between local and HF detection

In [10]:
overlap_repo_ids = list(all_one_row_per_repo[(all_local_flag == True) & (all_hf_flag == True)].index)
df_overlap_repos = pd.DataFrame({'repo_id': overlap_repo_ids})
got_overlap_unique = len(df_overlap_repos)
print("unique overlap repos:", got_overlap_unique)


unique overlap repos: 24


### Table - same Reposiotry-Same file vs Same Repository,different-file

In [ ]:
overlap_repo_id_set = set(df_overlap_repos['repo_id'])
local_files_per_repo = {}
hf_files_per_repo = {}
for repo_id in overlap_repo_id_set:
    local_files_per_repo[repo_id] = set()
    hf_files_per_repo[repo_id] = set()

for repo_id, one_type, one_task, details in all_the_malicious_details:
    if repo_id not in overlap_repo_id_set:
        continue

    start_to_cut = '/tmp/scan_workspace/' + repo_id.replace('/', '__') + '/'

    for one_file in (details.get('file_results', []) or []):
        path_of_this_file = one_file.get('file_path') or one_file.get('file_name') or ''
        if path_of_this_file.startswith(start_to_cut):
            path_of_this_file = path_of_this_file[len(start_to_cut):]
        for v in (one_file.get('verdicts', []) or []):
            if v.get('flagged') != True:
                continue
            if v.get('scanner') == 'hf_security_api':
                hf_files_per_repo[repo_id].add(path_of_this_file)
            if v.get('scanner') in local_scanners:
                local_files_per_repo[repo_id].add(path_of_this_file)

    # hf can flag the whole repo with no file - counts as one hf "file"
    hf_repo_verdict = details.get('hf_security_api')
    if isinstance(hf_repo_verdict, dict) and hf_repo_verdict.get('flagged') == True:
        hf_files_per_repo[repo_id].add('(whole repo - no file)')

same_file_repo_ids = set()
diff_file_repo_ids = set()
unique_same_file_pairs = set()
# each group to hold - [local only, hf only, both]
files_by_group = {'same file': [0, 0, 0], 'different file': [0, 0, 0]}

for repo_id in overlap_repo_id_set:
    local_files = local_files_per_repo[repo_id]
    hf_files = hf_files_per_repo[repo_id]
    files_in_both = local_files & hf_files

    if len(files_in_both) > 0:
        group_name = 'same file'
        same_file_repo_ids.add(repo_id)
        for one_file_name in files_in_both:
            unique_same_file_pairs.add((repo_id, one_file_name))
    else:
        group_name = 'different file'
        diff_file_repo_ids.add(repo_id)

    files_by_group[group_name][0] = files_by_group[group_name][0] + len(local_files - hf_files)
    files_by_group[group_name][1] = files_by_group[group_name][1] + len(hf_files - local_files)
    files_by_group[group_name][2] = files_by_group[group_name][2] + len(files_in_both)

print("same-repo, same-file repos:", len(same_file_repo_ids))
print("same-repo, different-file repos:", len(diff_file_repo_ids))
print("unique (repo, file) same-file matches:", len(unique_same_file_pairs))


same-repo, same-file repos: 7
same-repo, different-file repos: 17
unique (repo, file) same-file matches: 18


In [ ]:
# # one row per overlap repo, saved as a csv
# overlap_rows = []
# for repo_id in sorted(overlap_repo_id_set):
#     local_files, hf_files = local_files_per_repo[repo_id], hf_files_per_repo[repo_id]
#     overlap_rows.append({'repo_id': repo_id, 'group': 'same file' if repo_id in same_file_repo_ids else 'different file',
#                          'local_only_files': len(local_files - hf_files), 'hf_only_files': len(hf_files - local_files),
#                          'files_in_both': len(local_files & hf_files), 'local_files': '; '.join(sorted(local_files)),
#                          'hf_files': '; '.join(sorted(hf_files))})
# pd.DataFrame(overlap_rows).to_csv('due_diligence/overlap_same_vs_different_file.csv', index=False, encoding='utf-8-sig')
# print("saved", len(overlap_rows), "overlap repos")


<!-- ### Table 2a / 2b -- test-fixture vs real same-file matches

splits the unique same-file matches into the ones from the three GGUF test-fixture repos
(`dyingc/gguf-scanner-test-samples`, `Retr0REG/gguf-ssti`, `protecttors/sample-files`) and the organic ones. -->

## 3.5 - Risk-category classification (TAxonomy)


In [ ]:
rules_for_classifiation = [
    (r'oauth|jwt token|api key|secret detected|mailgun|heroku api key|telegram bot|hardcoded password|generic secret'
     r'|token detected|access key|private key|password detected|webhook detected|hockeyapp|bcrypt hash'
     r'|jwt secret|secret_key|hardcoded secret', 'Secrets'),

    # Malware/Suspicious Artifact'
    (r'bidirectional', 'Malware/Suspicious Artifact'),
    
    # Unsafe Serialization
    (r'pickle|unsafe pytorch load|torch\.load|deserializ|marshal|unsafe yaml|shelve|\bdill\b|read_stringnl_noescape', 'Unsafe Serialization'),
    
    # Other/Code Quality
    (r'revision pinning|mutable tag|pseudo-random', 'Other/Code Quality'),

    #  Security Misconfiguration - settings mistakes: debug mode on, listening on every ip, containers with too much power
    (r'debug=true|binding to all interfaces|host 0\.0\.0\.0|cors policy|permissive mask|widely permissive'
     r'|privilege escalation|privileged mode|writable root filesystem|node network|host ipc|host process id'
     r'|docker socket|workflow-level|without validating the password|hardcoded variable', 'Security Misconfiguration'),

    # Weak Crypto / Insecure Transport - weak hashes, old tls, nothing encrypted on the wire
    (r'md5|sha1|224-bit|ssl|\btls\b|certificate|verify=false|cipher|httpsconnection|httpconnectionpool'
     r'|unencrypted|insecure transport|not be encrypted|ftp|websocket|uuid version 1', 'Weak Crypto / Insecure Transport'),

    # Injection (SQL/Command)' 
    (r'sql injection|sql string|formatted sql query|parameterized quer', 'Injection (SQL/Command)'),
    # "shell" is in here because bandit says "starting a process with a shell" and never says "subprocess"
    (r'subprocess|os\.system|globals\(\)|url open|urllib|arbitrary file read|arbitrary code execution|command injection'
     r'|dynamic value.*url|partial executable path|with a shell|without a shell|shell=true|reverse shell', 'Injection (SQL/Command)'),
    # bandit writes "exec" and "eval" WITHOUT the brackets so \b instead of \(\) here
    (r'\beval\b|\bexec\b|literal_eval|import_module|are evaluated in|jinja2|xmlrpc|cross-site scripting|\bxss\b'
     r'|template injection|html string|httpresponse|open redirect|typecast|user-controlled request data'
     r'|arbitrary container|variable interpolation', 'Injection (SQL/Command)'),
    # xml attacks (XXE) and tar files that can write outside their folder
    (r'\bxml\b|xml external entity|tarfile|path traversal', 'Injection (SQL/Command)'),

    # Parser Failure
    (r'parse error|malformed or malicious|array too large|failed to parse|could not parse', 'Parser Failure'),
    
    # Dependency Vulnerability
    (r'cve-\d|dependency', 'Dependency Vulnerability'),
    
    # addd the HF scanners to Malware/Suspicious Artifact taxonomy
    (r'virustotal|protectai|jfrog|avscan|malware', 'Malware/Suspicious Artifact'),
]


compiled_rules_in_order = [(re.compile(pattern, re.IGNORECASE), category) for pattern, category in rules_for_classifiation]

# Default to these based on the scanners
default_category_for_each_scanner = {
    'bandit': 'Other/Code Quality', 'opengrep': 'Other/Code Quality',
    'pysentry': 'Dependency Vulnerability', 'pip_audit': 'Dependency Vulnerability', 'trivy': 'Dependency Vulnerability',
    'modelaudit': 'Parser Failure', 'hf_security_api': 'Malware/Suspicious Artifact',
}

def classify_one_message(scanner, message, item_type=''):
    msg = (message or '').lower()
    if scanner == 'trivy' and item_type == 'secret':
        return 'Secrets'
    if scanner == 'trivy' and item_type == 'misconfig':
        return 'Security Misconfiguration'
    if scanner in ['pip_audit', 'trivy', 'pysentry']:  # an EMPTY message means the tool failed/timed-out, not a vulnerability
        return 'Dependency Vulnerability' if msg.strip() else 'Scanner Failure (no finding)'
    if scanner == 'modelaudit' and 'suspicious metadata value' in msg:
        return 'Model Metadata Anomaly'
    for pattern, category in compiled_rules_in_order:    # first rule that matches
        if pattern.search(msg):
            return category
    return default_category_for_each_scanner.get(scanner, 'Other/Code Quality')


In [ ]:
def get_messages_of_one_verdict(verdict):
    d = verdict.get('details') or {}
    first_message = d.get('message') or d.get('issue_text') or d.get('description')
    msgs = [(str(first_message), '')] if first_message else []
    for it in [x for key in ['findings', 'issues', 'vulnerabilities', 'results'] for x in (d.get(key) or [])]:
        if isinstance(it, str):
            msgs.append((it, ''))
        elif isinstance(it, dict):
            m = (it.get('message') or it.get('description') or it.get('title') or it.get('check_id')
                 or it.get('id') or it.get('advisory') or it.get('summary') or it.get('rule_id'))
            if m:
                msgs.append((str(m), str(it.get('type') or '')))
    if verdict.get('scanner') == 'hf_security_api':
        msgs.append((str(d.get('status', 'flagged')), ''))
    return msgs or [('', '')]

# # test 1 - does the message finder work on each shape the scanners use?
# assert get_messages_of_one_verdict({'scanner': 'bandit', 'details': {'message': 'pickle used'}}) == [('pickle used', '')]
# assert get_messages_of_one_verdict({'scanner': 'trivy', 'details': {'results': [{'id': 'x', 'type': 'secret'}]}}) == [('x', 'secret')]
# print("checkpoint 1 ok - message finder works")

# # test 2 - does the classifier put known messages in the right category?
# assert classify_one_message('bandit', 'Pickle library used') == 'Unsafe Serialization'
# assert classify_one_message('trivy', 'anything', 'secret') == 'Secrets'
# print("checkpoint 2 ok - classifier works")


checkpoint 1 ok - message finder works
checkpoint 2 ok - classifier works


In [ ]:
repos_with_hf_files_in_the_rescan = {repo_id for repo_id, one_type, one_task, details in all_the_malicious_details
    if one_type == 'HFscan' and any(f.get('flagged') == True and 'hf_security_api' in (f.get('flagged_by') or [])
                                    for f in (details.get('file_results') or []))}
print("repos that got their HF files from the rescan db:", len(repos_with_hf_files_in_the_rescan))

def build_risk_rows():
    rows = []   # (repo_id, file_key, risk_category, scanner, what_was_flagged)
    main_files_done, local_files_done, hf_files_done, over_cap_done, repo_verdict_done = set(), set(), set(), set(), set()

    for repo_id, one_type, one_task, details in all_the_malicious_details:
        file_results = details.get('file_results') or []
        start_to_cut = '/tmp/scan_workspace/' + repo_id.replace('/', '__') + '/'

        # repos over the 30 GB cap: no file_results, just a flagged_files list from HF
        if one_type != 'HFscan' and len(file_results) == 0:
            for one_path in (details.get('flagged_files') or []):
                file_key = repo_id + ' :: ' + one_path.removeprefix(start_to_cut)
                if file_key not in over_cap_done:
                    over_cap_done.add(file_key)
                    rows.append((repo_id, file_key, classify_one_message('hf_security_api', 'unsafe'), 'hf_security_api', 'unsafe'))

        for one_file in file_results:
            if one_file.get('flagged') != True:
                continue
            file_key = repo_id + ' :: ' + (one_file.get('file_path') or one_file.get('file_name') or '').removeprefix(start_to_cut)
            if one_type != 'HFscan':
                if file_key in main_files_done:
                    continue
                main_files_done.add(file_key)

            for v in (one_file.get('verdicts') or []):
                scanner = v.get('scanner')
                if v.get('flagged') != True:
                    continue
                if one_type == 'HFscan' and scanner != 'hf_security_api' and file_key in local_files_done:
                    continue
                if one_type != 'HFscan' and scanner in local_scanners:
                    local_files_done.add(file_key)
                if scanner == 'hf_security_api':
                    if one_type == 'HFscan' and file_key in hf_files_done:
                        continue
                    hf_files_done.add(file_key)
                for message_text, message_type in get_messages_of_one_verdict(v):
                    rows.append((repo_id, file_key, classify_one_message(scanner, message_text, message_type), scanner, message_text.split(':')[0].strip()[:80]))

        # HF sometimes gives one verdict for the whole repo with no file - one row, once per repo
        hf_repo_verdict = details.get('hf_security_api')
        if (isinstance(hf_repo_verdict, dict) and hf_repo_verdict.get('flagged') == True
                and repo_id not in repos_with_hf_files_in_the_rescan and repo_id not in repo_verdict_done):
            repo_verdict_done.add(repo_id)
            hf_repo_status = str((hf_repo_verdict.get('details') or {}).get('status', 'flagged'))
            rows.append((repo_id, None, classify_one_message('hf_security_api', hf_repo_status), 'hf_security_api', hf_repo_status))
    return rows

df_repo_file = pd.DataFrame(build_risk_rows(), columns=['repo_id', 'file_key', 'risk_category', 'scanner', 'what_was_flagged'])
print("rows:", len(df_repo_file), "(was 482622 when checked on 2026-10-27)")

# # test - every row has a category and every repo really is a malicious repo
# assert df_repo_file['risk_category'].notna().all()
# assert set(df_repo_file['repo_id']) <= {row[0] for row in all_the_malicious_details}
# print("checkpoint 3 ok - all rows have a category and a malicious repo")


In [28]:
print("=== per risk category, all scanners together (a file/repo counts once per category) ===")
per_category_table = pd.DataFrame({
    'detections': df_repo_file.groupby('risk_category').size(),
    'files': df_repo_file.groupby('risk_category')['file_key'].nunique(),
    'repos': df_repo_file.groupby('risk_category')['repo_id'].nunique(),
}).sort_values('detections', ascending=False)
per_category_table.loc['Total (distinct)'] = [len(df_repo_file), df_repo_file['file_key'].nunique(), df_repo_file['repo_id'].nunique()]
print(per_category_table.to_string())

=== per risk category, all scanners together (a file/repo counts once per category) ===
                                  detections   files  repos
risk_category                                              
Other/Code Quality                    128199   28539  13591
Parser Failure                        111401   55720  15332
Dependency Vulnerability              106974    2216   1233
Model Metadata Anomaly                 63704   32337   8064
Unsafe Serialization                   24671    6978   4200
Secrets                                15905    5902   4107
Malware/Suspicious Artifact            13071    4557   2535
Injection (SQL/Command)                10816    3189   1630
Scanner Failure (no finding)            4118    3740   2853
Security Misconfiguration               2574     660    288
Weak Crypto / Insecure Transport        1189     441    244
Total (distinct)                      482622  103580  35361


In [ ]:
# what was flagged inside each category: the top 20 kinds of finding (by repos), saved as a csv
# what_was_flagged = df_repo_file.groupby(['risk_category', 'what_was_flagged']).agg(
#     scanners=('scanner', lambda s: ', '.join(sorted(set(s)))),
#     repos_with_at_least_one=('repo_id', 'nunique'), files=('file_key', 'nunique'),
#     findings=('repo_id', 'size'), example_repo=('repo_id', 'first')).reset_index()
# what_was_flagged = what_was_flagged.sort_values(['risk_category', 'repos_with_at_least_one'], ascending=[True, False])
# what_was_flagged.insert(1, 'rank', what_was_flagged.groupby('risk_category').cumcount() + 1)
# what_was_flagged = what_was_flagged[what_was_flagged['rank'] <= 20].rename(columns={'risk_category': 'category'})
# what_was_flagged.to_csv('due_diligence/what_was_flagged_by_category_new.csv', index=False, encoding='utf-8-sig')
# print("saved", len(what_was_flagged), "rows")
# print(what_was_flagged.groupby('category').size())
